# Explanation benchmark — run and results

The long run belongs in a terminal, not Jupyter (same rule as `eval_conditions.py` / `train_conditions.py`). From the repo root, in the project venv:

```bat
python scripts\test_xai_eval.py                                        :: 27 known-answer checks, seconds, CPU
python scripts\run_xai_eval.py --n 8 --n-stability 2 --arb-limit 32    :: smoke, a few minutes
python scripts\run_xai_eval.py                                         :: full: C3 spatial study + 4-model panel
```

* The stages can run separately (`--stages spatial` / `--stages arbitration`); `REPORT.md` always includes whatever is on disk.
* Stage 1 checkpoints every 25 clips. If interrupted, `--resummarise` rebuilds every table and figure from what finished (the report marks it PARTIAL).
* `--amp` roughly halves the time; scores then drift ~1e-3 from `eval_conditions.py`, so the per-clip consistency check will print warnings.
* No GPU: `--cpu-fallback` (n=32, no KernelSHAP).

In [ ]:
from pathlib import Path
import json, pandas as pd
from IPython.display import Markdown, Image, display
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "results" / "xai"          # results/xai_demo for the plumbing run
print(OUT, "exists" if OUT.exists() else "MISSING - run scripts/run_xai_eval.py first")
def fig(name):
    p = OUT / "figures" / name
    if p.exists(): display(Image(str(p)))

## 1. Spatial faithfulness

Every method against random and the centre prior, under blur, mean-face and roll fills. `d_del_vs_random > 0` = deletion falls faster than random; `beats_random` needs the 95% CI to exclude zero. The winner must beat random under **both** blur and mean-face; roll is CFX's in-distribution fill, kept for comparability with `cfx_eval.py`.

In [ ]:
s = json.load(open(OUT / "spatial_summary.json"))
t = pd.read_csv(OUT / "spatial_faithfulness.csv")
display(t[["method","fill","del_auc","ins_auc","comp_mean","suff_abs","flip_frac",
           "d_del_vs_random","d_del_CI","p_del","d_del_vs_centre","p_centre","beats_random","borda"]])
print("winner:", s["winner"])
print("fill agreement (Kendall tau):", s["fill_rank_agreement_kendall_tau"])
print("controls:", json.dumps(s["controls"], indent=1))
print("modality gate:", json.dumps(s["modality_gate"], indent=1))

In [ ]:
for f in ["fig_deletion_insertion.png", "fig_gain_vs_random.png", "fig_example_maps.png"]: fig(f)

## 2. Stability · 3. Model-randomisation sanity check (ρ near 0 is a pass)

In [ ]:
display(pd.read_csv(OUT / "stability.csv"))
p = OUT / "sanity.csv"
if p.stat().st_size > 5: display(pd.read_csv(p, header=[0, 1], index_col=0))

## 4. Temporal faithfulness and counterfactuals

In [ ]:
print(json.dumps(s["temporal"], indent=1)); print(json.dumps(s["counterfactuals"], indent=1))
fig("fig_temporal_spectral.png")

## 5. Does faithfulness predict correctness beyond confidence?

In [ ]:
print(json.dumps(s["faithfulness_vs_correctness"], indent=1))

## 6. Reliability matrix · 7. Arbitration vs single models

In [ ]:
if (OUT / "arbitration_summary.json").exists():
    display(pd.read_csv(OUT / "reliability_matrix.csv").round(3))
    display(pd.read_csv(OUT / "arbitration_comparison.csv").round(4))
    a = json.load(open(OUT / "arbitration_summary.json"))
    for k in ("best_single_by_val", "val_cv_nll_platt", "stack_weight_share_by_model",
              "routing_prediction_held", "decision"):
        print(f"{k}: {a[k]}")
    fig("fig_reliability_matrix.png")

## Full report (auto-generated)

In [ ]:
display(Markdown((OUT / "REPORT.md").read_text(encoding="utf-8")))